### Processing the data

#### data wrangling and cleaning

In [ ]:
# import dependencies
import pandas as pd

# data cleaning process
df = pd.read_csv('resources/exoplanet_data.csv')
# df.head()

# drop all null columns and rows
column_clean = df.dropna(axis='columns', how='all')
# column_clean.head()
df_clean = column_clean.dropna()
df_clean.head(9)

#### pre-processing data using sklearn

In [ ]:
from sklearn.model_selection import train_test_split
# create x and y variables
X = df_clean.drop('koi_disposition', axis=1)
y = df_clean['koi_disposition']

# split data to create training and testing data
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, stratify=y)

from sklearn.preprocessing import MinMaxScaler
# scale x values
X_scaler = MinMaxScaler().fit(X_train)
X_train_scaled = X_scaler.transform(X_train)
X_test_scaled = X_scaler.transform(X_test)

### Model building

#### model training using RFC

In [ ]:
# start training the datasets
from sklearn.ensemble import RandomForestClassifier
classifier = RandomForestClassifier(n_estimators=200)
classifier.fit(X_train_scaled, y_train)

model_training = round(classifier.score(X_train_scaled, y_train)*100,2)
model0_accuracy = round(classifier.score(X_test_scaled, y_test)*100,3)

print(f'training model score = {model_training} %')
print(f'testing model score = {model0_accuracy} %')

### feature selection

In [ ]:
# feature selection using RF
fn = X.columns.tolist()
feature_select = sorted(zip(classifier.feature_importances_, fn), reverse=True)
features_sorted = pd.DataFrame(feature_select, columns=['score', 'feature']).set_index('feature')
features_sorted.reset_index().head(9)

In [ ]:
# select the required features to be used for further model tuning
approved_features = []
for i in feature_select:
    if i[0] > 0.01:
        approved_features.append(i[1])

In [ ]:
# assign the new datasets variables
X_train_new = X_train[approved_features]
X_test_new = X_test[approved_features]

scaler = MinMaxScaler()
X_scaler = scaler.fit(X_train_new)

X_train_scaled = X_scaler.transform(X_train_new)
X_test_scaled = X_scaler.transform(X_test_new)

# training the new model using RFC
new_model = RandomForestClassifier(n_estimators=200)
new_model.fit(X_train_scaled, y_train)

new_model_training_score = round(new_model.score(X_train_scaled, y_train)*100,2)
new_model_features_score = round(new_model.score(X_test_scaled, y_test)*100,2)

print(f'training model score = {new_model_training_score} %')
print(f'testing model score = {new_model_features_score} %')

### Model tuning process

In [ ]:
# model tuning
# create gridsearchCV
from sklearn.model_selection import GridSearchCV

model_2 = RandomForestClassifier(random_state=42)
param_grid = {
    'max_features': ['auto', 'sqrt', 'log2'],
    'n_estimators': [200, 600, 1200, 1400],
    'max_depth': [14,15,16,17,18,None]
}

grid = GridSearchCV(model_2, param_grid, cv=5, verbose=3, n_jobs=-1)

In [ ]:
# fit model with gridsearchCV
fit_model = grid.fit(X_train_scaled, y_train)

In [ ]:
# tune model params
n_estimators = grid.best_params_['n_estimators']
max_features = grid.best_params_['max_features']
max_depth = grid.best_params_['max_depth']
criteria = 'entropy'

# tune the model
t_model = RandomForestClassifier(
    max_features=max_features,n_estimators=n_estimators,max_depth=max_depth,
    criterion=criteria, random_state=42)
t_model.fit(X_train_scaled, y_train)

model_3 = round(t_model.score(X_train_scaled, y_train)*100,2)
tuned_model_score = round(t_model.score(X_test_scaled, y_test)*100,2)

print(f'training model score = {model_3} %')
print(f'testing  model score = {tuned_model_score} %')

### Model prediction

In [ ]:
# model predictions
predictions = t_model.predict(X_test_scaled)
sorting_y_test_data = y_test.unique().tolist()

overall = {'real': y_test, 'predictions':predictions}
# create a dataframe for the predicted and actual values to compare against
comparison_df = pd.DataFrame(overall).set_index('real')
comparison_df.reset_index().head(9)

### evaluating the models

In [ ]:
# make evaluations of the models
eval = {'model tested': ['root model', 'selected features model', 'tuned model'],
    'accuracy': [f'{model0_accuracy}%', f'{new_model_features_score}%', f'{tuned_model_score}%']}

eval_df = pd.DataFrame(eval).set_index('model tested').reset_index()
eval_df

In [ ]:
# save the model scores to a csv 
eval_df.to_csv('resources/RFC_scores.csv')


import joblib
filename = 'ML_models/RFC.sav'
joblib.dump(t_model, filename)